# CRIVO — Salto do 17M, fase 2: pré-treino continuado

**Diagnóstico.** Nove rodadas puseram cabeças (ponte, pointer, cópia, parada, adaptador) sobre a base congelada, e nenhuma passou. A sonda de cópia em contexto (`[A][B] … [A] → [B]`) mostrou o porquê: o 17M **acerta só 27%** de uma sequência que acabou de ver. Essa capacidade (*induction heads*) se forma no pré-treino, numa mudança de fase que a literatura situa em bilhões de tokens; a base viu ~0,33 bilhão. As cabeças tentavam compensar de fora uma habilidade que falta dentro.

**O que este notebook faz.** Continua o pré-treino **do mesmo modelo**, com os mesmos dados próprios (Wikipédia pt, diálogos humanos do OASST2, acervo do CRIVO) e o mesmo tokenizador:
- parte de `CRIVO/transformer-leitor/checkpoint.pt` (passo 20.000), **só leitura**;
- grava tudo em `CRIVO/transformer-17m-continuo/` (checkpoint único, base exportada no fim);
- +40.000 passos (~655 milhões de tokens), re-aquecimento curto e cosseno;
- 5% das janelas recebem um trecho repetido da própria janela (repetição favorece a formação da cópia);
- **sonda de cópia a cada 2.500 passos** — a métrica principal desta fase.

**Como rodar.** *Executar tudo*. Ele pausa sozinho em ~3h20 com checkpoint; na próxima sessão, *Executar tudo* de novo e ele retoma (são ~3 sessões na T4). Os dados ficam em cache no Drive (`CRIVO/dados-17m/`) se houver espaço, para não baixar a cada sessão.

**Metas da fase:** cópia top-1 ≥ 80% (era 27%) e nome inventado na 2ª menção ≤ 1,0 nat (era ~2,4), sem piorar a validação. Nada vai para o chat nesta fase: `ativo_no_chat=false`, `promovido=false`.


In [ ]:
# Célula 0 — GPU, Drive e checkpoint de origem
import os, sys, json, shutil, subprocess, hashlib, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, flush=True)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

CRIVO = Path("/content/drive/MyDrive/CRIVO")
ORIGEM = CRIVO / "transformer-leitor" / "checkpoint.pt"      # pré-treino de 20.000 passos (só leitura)
SAIDA = CRIVO / "transformer-17m-continuo"                    # tudo desta fase fica aqui
CHECKPOINT = SAIDA / "checkpoint.pt"
BASE = SAIDA / "base"
CACHE = CRIVO / "dados-17m"                                   # treino.bin/validacao.bin, para não refazer a cada sessão
PASSOS_EXTRA = 40000      # ~655 milhões de tokens extras; na T4 ~1,2 passo/s ≈ 9h20 em 3 sessões
MAX_HORAS = 3.3           # pausa com checkpoint e retoma na próxima sessão

if not ORIGEM.is_file() and not CHECKPOINT.is_file():
    raise SystemExit(f"ERRO: não encontrei {ORIGEM}")
SAIDA.mkdir(parents=True, exist_ok=True)
livre = shutil.disk_usage("/content/drive/MyDrive").free / 1024**3
if CHECKPOINT.is_file():
    print(f"Checkpoint para retomar: {CHECKPOINT.stat().st_size/1024**2:.0f} MiB (a origem não é necessária)", flush=True)
else:
    print(f"Origem: {ORIGEM} ({ORIGEM.stat().st_size/1024**2:.0f} MiB)", flush=True)
print(f"Drive livre: {livre:.1f} GiB", flush=True)
print("Retomando" if CHECKPOINT.is_file() else "Primeira sessão", "→", CHECKPOINT, flush=True)


In [ ]:
# Célula 1 — código do CRIVO (branch do experimento)
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1", "pyarrow"], check=True)
print("commit:", COMMIT, flush=True)


In [ ]:
# Célula 2 — dados próprios (Wikipédia pt + OASST2 + acervo), com o tokenizador da base
TOKENIZER_SHA = "6e37a7508def06d31f648a1fb816a67df99e806d450051b611a2406f044dbe7e"
DADOS = Path("/content/crivo-dados")
DADOS.mkdir(parents=True, exist_ok=True)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def rodar(cmd):
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linha in proc.stdout:
        print(linha, end="", flush=True)
    if proc.wait() != 0:
        raise SystemExit(f"ERRO: comando falhou ({proc.returncode})")

tok_base = REPO / "experimentos/pesos_base/leitor_transformer/tokenizer.json"
if sha(tok_base) != TOKENIZER_SHA:
    raise SystemExit("ERRO: tokenizer do repositório difere do da base")
shutil.copyfile(tok_base, DADOS / "tokenizer.json")

if all((CACHE / n).is_file() for n in ("treino.bin", "validacao.bin", "manifesto.json")):
    print("Usando dados em cache no Drive:", CACHE, flush=True)
    for n in ("treino.bin", "validacao.bin", "manifesto.json"):
        shutil.copyfile(CACHE / n, DADOS / n)
else:
    t0 = time.time()
    rodar([sys.executable, "scripts/pretreinar_leitor_16m.py", "--etapa", "dados", "--dados", str(DADOS)])
    rodar([sys.executable, "scripts/pretreinar_leitor_16m.py", "--etapa", "tokenizar", "--dados", str(DADOS)])
    print(f"dados prontos em {(time.time()-t0)/60:.1f} min", flush=True)
    tamanho = sum((DADOS / n).stat().st_size for n in ("treino.bin", "validacao.bin")) / 1024**3
    if shutil.disk_usage("/content/drive/MyDrive").free / 1024**3 > tamanho + 3:
        CACHE.mkdir(parents=True, exist_ok=True)
        for n in ("treino.bin", "validacao.bin", "manifesto.json"):
            shutil.copyfile(DADOS / n, CACHE / n)
        print(f"Cache gravado no Drive ({tamanho:.2f} GiB): próximas sessões pulam o download.", flush=True)
    else:
        print("Drive sem espaço para o cache; a próxima sessão baixa de novo.", flush=True)
if sha(DADOS / "tokenizer.json") != TOKENIZER_SHA:
    raise SystemExit("ERRO: tokenizer dos dados difere do da base")
print("treino:", (DADOS / "treino.bin").stat().st_size // 2, "tokens | validação:",
      (DADOS / "validacao.bin").stat().st_size // 2, "tokens", flush=True)


In [ ]:
# Célula 3 — pré-treino continuado (pausa sozinho em MAX_HORAS; rode de novo para continuar)
rodar([sys.executable, "scripts/continuar_pretreino_17m.py", "--etapa", "treinar",
       "--dados", str(DADOS), "--origem", str(ORIGEM), "--checkpoint", str(CHECKPOINT),
       "--passos-extra", str(PASSOS_EXTRA), "--max-horas", str(MAX_HORAS)])


In [ ]:
# Célula 4 — andamento da sonda de cópia e exportação quando terminar
estado = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
hist = estado["historico"]
print(f"passo {estado['passo']} de {estado['passo_inicial'] + PASSOS_EXTRA}\n", flush=True)
print(f"{'passo':>7} {'tokens extra':>13} {'cópia top-1':>12} {'2ª vez (nats)':>14} {'nome 2ª menção':>15} {'validação':>10}")
for h in hist:
    print(f"{h['passo']:>7} {h['tokens_extra']/1e6:>11.0f} M {100*h['copia_top1']:>11.1f}% {h['perda_2a_vez']:>14.2f} "
          f"{h['nome_2a_mencao']:>15.2f} {h['validacao']:>10.4f}")
del estado

if hist and hist[-1]["passo"] >= hist[0]["passo"] + PASSOS_EXTRA:
    rodar([sys.executable, "scripts/continuar_pretreino_17m.py", "--etapa", "exportar", "--dados", str(DADOS),
           "--checkpoint", str(CHECKPOINT), "--base", str(BASE), "--passos-extra", str(PASSOS_EXTRA)])
    rel = json.loads((BASE / "relatorio.json").read_text(encoding="utf-8"))
    print("\nBASE NOVA:", BASE, "| pesos sha256:", rel["pesos_sha256"], flush=True)
    print("Metas da fase 2: cópia top-1 ≥ 80% e nome na 2ª menção ≤ 1,0 nat, sem piorar a validação.", flush=True)
    print("ativo_no_chat=false | promovido=false — a fase 3 (diálogo) parte desta base.", flush=True)
else:
    print("\nAinda não terminou: rode de novo o notebook (Executar tudo) na próxima sessão; ele retoma sozinho.", flush=True)
